<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/main/src/eval/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Libraries

In [1]:
!pip install -q transformers>=4.45.0 accelerate torch torchvision pillow scikit-learn tqdm chess cairosvg python-Levenshtein

Importing Libraries

In [2]:
import os
import sys
import json
import subprocess
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import AutoModelForImageTextToText, AutoProcessor

Setting up environment

In [4]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent

# 2. Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

print(f"Setup Complete. REPO_ROOT: {repo_root}")

# 3. Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 4. Import custom project modules cleanly (from data)
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

# 5. Import evaluation modules/utilities from src/eval/utilities.py
from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_1,
)

print("All custom modules and eval utilities imported successfully!")

Repository directory already exists at: /content/BigDataAndTextMiningProject
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: Tesla T4


ImportError: cannot import name 'evaluate_chessboard_model_task_1' from 'eval.utilities' (/content/BigDataAndTextMiningProject/src/eval/utilities.py)

Dowloading dataset from HuggingFace repo

In [ ]:
print("Verifying Authentication to Hugging Face...")
notebook_login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

# Specifichiamo esplicitamente 'imagefolder' per dire a Hugging Face
# di leggere la struttura basata su metadata.jsonl che abbiamo generato
dataset_task1 = load_dataset(dataset_name)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Loading Baseline Model

In [ ]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

Test with baseline

In [ ]:
# 1. Grab the first test sample directly from your loaded Hugging Face dataset variable
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]

# 2. Use the image already downloaded and loaded by Hugging Face dataset
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# 3. Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# 4. Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# 5. Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model.device)

# 6. Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model.generate(**model_inputs, max_new_tokens=128)

# 7. Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

The zero-shot evaluation of the base model reveals a complete failure in spatial reasoning and fine-grained geometric extraction. Instead of generating a valid FEN string corresponding to the chessboard pieces, the model hallucinates a random mix of board coordinates (e.g., a-h, 1-8) and structural patterns.

This behavior empirically confirms our research hypothesis: standard Vision-Language Models (VLMs) lack the intrinsic capability to accurately parse structured grid-based board states without task-specific supervision. This baseline failure validates the necessity of applying Supervised Fine-Tuning (SFT) via LoRA to bridge the gap between visual input and precise FEN notation.

In [ ]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_1(
    model=model,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head)

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)